# spm_reml_sc explained

## Overview of the GLM

The previous notebook presented the details of vRSA at a conceptual level. In this notebook, we will dig into how the algorithm we use to implement it works. As briefly mentioned in the previous notebook, and more extensively described in our paper, vRSA is based on the generalized linear model formulae:

$$
\mathbf{Y} = \mathbf{X}\mathbf{\beta} + \mathbf{Z}\mathbf{U} + \mathbf{E}
$$

In this case, all the symbols are bold because we are dealing with matrices. The data $\mathbf{Y}$ are of dimension $M \times P$, where $M$ are the number of measurements (i.e. trials, trials x time points...), $P$ is the number of measurement channels (i.e. voxels, eeg channels...). 


In a linear models, we are interested in estimating the following things. For the fixed effect, we are interested in estimating their means:

$$
\mathbf{\beta} \sim \mathcal{N}(\mu, \sigma_{v}^2)
$$

For the random effects, the goal is to estimate the variance covariance matrix $G$:

$$
\mathbf{U} \sim \mathcal{N}(0, G)
$$

Finally, there is one distributional assumption that has to be made, which is that the errors are independent and identically distributed (i.i.d.): 

$$
\mathbf{E} \sim \mathcal{N}(0, S \otimes I_M)
$$

In this case, since we are dealing with a multivariate model, the i.i.d. assumption allows for spatial covariance across channels $S$. The equation above is basically saying, except for the spatial covariance across channels ($S$), the error should be independent and identically distribution once we account for the fixed and random effects.

## Estimating variance components

As stated above, the goal in a mixed model is to estimate three things:
- $\beta$: a $w$ fixed effects by $p$ channels/voxels
- $\mathbf{G}$: a $m$ by $m$ matrix, capturing the covariance between the random effects of the data 
- $\mathbf{S}$: the $p$ by $p$ matrix capturing the spatial covariance of the data

In the specific case of a vRSA and pattern component modelling, we are not interested in the fixed effects parameters, what we are after are the variance parameters, specificially $G$. Note that we are not either interested in the $\mathbf{U}$ parameters of the model itself either. We are only indirectly interested in them. That is because if we were able to estimate the parameters $\mathbf{U}$ exactly, the covariance of the random effects would be:

$$
\mathbf{G} = \mathbf{U}\mathbf{U}^T  
$$

::: {.callout-note}
When using a mixed linear model of the form  $\mathbf{Y} = \mathbf{X}\mathbf{\beta} + \mathbf{Z}\mathbf{U} + \mathbf{E}$, we are stating that we believe the generative process for the data $\mathbf{Y}$ we have observed is a linear model, such that the data are generated by a process in which the fixed effects in $\mathbf{X}$ multiplied by their $\mathbf{\beta}$ weight, to which we add the random effects in $\mathbf{Z}$ multiplied by $\mathbf{U}$. What we really want to know are the true values of the $\mathbf{\beta}$ and $\mathbf{U}$ (or $\mathbf{G}$), but we can't. We can only estimate them from the data that we have. We make the distinction between the true parameters $\mathbf{\beta}$ and $\mathbf{U}$ and the estimated ones by adding a hat $\mathbf{\hat{\beta}}$ and $\mathbf{\hat{U}}$. They are different because the true parameters are supposed to be fixed, but what we estimates are only approximations of the ground truth that depends on the data we have collected. If we collect a whole new data set, replicating exactly the one we have, the estimated values of $\mathbf{\hat{\beta}}$ and $\mathbf{\hat{U}}$ would be slightly different each time. Importantly, if we knew the ground truth of $\mathbf{U}$, then obtaining the covariance of those random effects would be indeed as simple as $\mathbf{G}\mathbf{G}=\mathbf{U}\mathbf{U}^T$
:::

So it would seem like the most straight forward way to compute the covariance matrix $\mathbf{G}$ would be to first estimate $\hat{\mathbf{U}}$ and then multiply that by its transpose to obtain $\hat{G}$. There is nothing wrong per se with trying to estimate $\mathbf{G}$ that way. An alternative however is to try to estimate $\mathbf{G}$ directly from the data, since it's what we are interested in. 

Since we know that $\mathbf{G}=\mathbf{U}\mathbf{U}^T$, we can rewrite our generative model to try to show $\mathbf{G}$. We can express the generative model in terms of the so-called second moment of the data:

$$
\begin{align}
\mathbf{Y}\mathbf{Y}^T =& \mathbf{X}\mathbf{\beta}\mathbf{\beta}^T\mathbf{X}^T + \mathbf{Z}\mathbf{U}\mathbf{U}^T\mathbf{Z}^T + \mathbf{E}\mathbf{E}^T
=& \mathbf{X}\mathbf{\beta}\mathbf{\beta}^T\mathbf{X}^T + \mathbf{Z}\mathbf{G}\mathbf{Z}^T + \mathbf{E}\mathbf{E}^T
\end{align}
$$

This is the generative model of the covariance structure of the data, which is saying "the covariance of the observed data $\mathbf{Y}\mathbf{Y}^T$ (i.e. sample covariance) is equal to the covariance of the fixed effect, the covariance of the random effects, and the noise covariance". And so based on this model, instead of trying to first estimate $\mathbf{U}$ to get to the covariance, we can try to directly estimate the covariance of the random effects $\mathbf{G}$ from the covariance of the observed data $\mathbf{Y}\mathbf{Y}^T$. 

One way we could do so is by using the same algorithm used to estimate the fixed effets: Maximum likelihood estimation (MLE). We won't go into too much details, but there are essentially two issues when using MLE to estimate the random effects covariance. Both issues relate to the fact that the maximum likelihood estimate (MLE) formulae of $\mathbf{\hat{G}}$ requires estimates of $\mathbf{\hat{\beta}}$. The first important issue is that taking the MLE of a $\mathbf{G}$ requires estimating the fixed effects $\mathbf{\hat{\beta}}$ but ignores the fact that degrees of freedom have been lost in that estimation. In addition, calculating the MLE of $\mathbf{G}$ depends on $\mathbf{\hat{\beta}}$. But since the estimates of $\mathbf{\hat{\beta}}$ are by definition a bit off (they are only estimates, not the true value of the parameters), the estimates of $\mathbf{G}$ will be dependent on the error in  $\mathbf{\hat{\beta}}$, which is less than ideal, especially if we could avoid it (you can find a more thorough description of the issues [here](https://xiuming.info/docs/tutorials/reml.pdf)).

## Restricted maximum likelihood estimate (ReML)
And avoid it we can. There is an alternative way in which we can estimate $\hat{\mathbf{G}}$ such that it does not depend on the fixed effects. At a conceptual level, what we do is that we project the observed covariance matrix $\mathbf{Y}\mathbf{Y}^T$ in a space that is orthogonal to the fixed effects $\mathbf{X}$, such that the estimation of the parameters $\hat{G}$ depends only on the residuals that do not reflect the fixed effects. The way this is done is by defining a matrix, which we will call $\mathbf{A}$, such that when we multiply $\mathbf{X}$ by it, we get 0: $\mathbf{A}\mathbf{X}=0$. There is in fact one such matrix, which we call the residual maker matrix. You can find some information about it [here](https://en.wikipedia.org/wiki/Projection_matrix). So instead of defining a likelihood function as $\mathcal{L}(\mathbf{\beta}, \mathbf{G}, \mathbf{E}|\mathbf{Y}, \mathbf{X}, \mathbf{Z})$, we define the likelihood function as:

$$
\mathcal{L}(\mathbf{G}, \mathbf{E}|\mathbf{A}\mathbf{Y}, \mathbf{Z})
$$

That is, we are defining a likelihood of the parameters not given the data $\mathbf{Y}$ but given the data projected in a space that is orthogonal to the fixed effects $\mathbf{A}\mathbf{Y}$. And this is what we call Restricted Maximum Likelihood, or ReML, because it is a likelihood function that is restricted to the subspace of the data that is orthogonal to the fixed effects (see [here]() for an illustration of the idea of orthogonal projection). 

We have not defined the ReML function yet. But first, let's remind ourselves what we are trying to do: find the values of $\mathbf{G}$ that maximize whatever that function will be. If you remember your calculus classes, you will know that what that actually means is that we have to find some values for the parameters where the derivative is equal to 0. And importantly, while we have said that the ReML function finds the estimates given the data that have been multiplied by a projection matrix $\mathbf{A}$, we don't actually ever need to explicitely do $\mathbf{A}\mathbf{Y}$ at any point when calculating the values for $\mathbf{G}$. That is because when we derive the likelihood function, we can make use of many clever mathematical tricks to actually remove that bit and define the likelihood function as something that's a bit easier. We will skip the details of how we get to the ReML function, but the idea is basically the same as for defining the MLE function, but then adding the multiplication by the projection matrix $\mathbf{A}$, and then rearranging terms to get whatever formulation is the easiest to work with. You can find all the relevant details of the derivation of the restricted likelihood function [here](http://www.leg.ufpr.br/~eder/Variance%20Components.pdf) from page 249 onwards. 

Here we skip ahead to the log likelihood function directly:

$$
\mathcal{L}(\theta|\mathbf{A}^Ty) = -\frac{1}{2}(N-k)log(2\pi) -\frac{1}{2}log|\mathbf{C}|- \frac{1}{2} log |\mathbf{X}^T \mathbf{C}^{-1} \mathbf{X}| - \frac{1}{2}(\mathbf{Y}-\mathbf{X}\hat{\beta})^T\mathbf{C}^{-1}(\mathbf{Y}-\mathbf{X}\hat{\beta})
$$

We write $\theta$ as a parametrization of $\mathbf{G}$ and $\sigma^2$, but you can read it as being the same. If that's not clear, it doesn't matter, because it won't really play a big role below, just pretend they are the same. Note that this is not the restricted likelihood function, but the log of the restribted likelihood. For simplicity, we still denote it as $\mathcal{L}$, but it is important to remember the distinction. And the reason why we use the log of the likelihood is because it simplifies the math down the line (division are substraction and things like that), while still preserving all the relevant properties: whichever values minimize the log of the likelihood also by definition minimize the log thereof. We will refer to the function above as the likelihood below, but remember that it is in fact the log likelihood. 

The $\mathbf{C}$ above is the overall covariance of the data, which is defined as:

$$
\mathbf{C} = \mathbf{Z}\mathbf{G}\mathbf{Z}^T + \sigma^2 \mathbf{I}
$$

## ReML simplification

The first thing to remember is that with ReML, what we want to find are the estimates of the variance components that maximize the (restricted) likelihood function. This in turns means findings the values for $\mathbf{G}$ such that the derivative of the ReML function is equal to 0. That means that we actually don't even need to calculate what the (log) likelihood function is equal to for various values of $\mathbf{G}$, we only need to calculate the derivative and find where it's equal to 0 (i.e. solve $\mathcal{L}(\mathbf{G}|A^Ty)'=0$). What that in turns means is that any parts of the equation that don't contain $\mathbf{G}$ can be ignored, cause these are just constant that won't have an impact on the derivative. So we can rewrite the equation above ingoring the constant:

$$
\mathcal{L}(\theta|\mathbf{A}^T\mathbf{Y}) = -\frac{1}{2}log|\mathbf{C}|- \frac{1}{2} log |\mathbf{X}^T \mathbf{C}^{-1} \mathbf{X}| - \frac{1}{2}(\mathbf{Y}-\mathbf{X}\hat{\beta})^T\mathbf{C}^{-1}(\mathbf{Y}-\mathbf{X}\hat{\beta})
$$

Another important thing to notice is the last part of the equation:

$$
(\mathbf{Y}-\mathbf{X}\hat{\beta})^TC^{-1}(\mathbf{Y}-\mathbf{X}\hat{\beta})
$$


Parts of it look very much like the residuals of a linear model: $\mathbf{Y}-\mathbf{X}\hat{\beta}$ is basically the observed data, minus the fitted data, so only error (i.e. residuals) remains. To simplify the notation, we can define $r=\mathbf{Y}-\mathbf{X}\hat{\beta}$, so the above is simply $r^TC^{-1}r$. We can develop the equation to get the following:

$$
r^T\mathbf{C}^{-1}r = \mathbf{Y}^T \mathbf{C}^{-1} \mathbf{Y} - \mathbf{Y}^T \mathbf{C}^{-1} \mathbf{X}\hat{\beta} - \hat{\beta}^T\mathbf{X}^T \mathbf{C}^{-1}\mathbf{Y} + \hat{\beta}^T\mathbf{X}^T \mathbf{C}^{-1} \mathbf{X}\hat{\beta}
$$

Importantly, under ReML, the $\hat{\beta}$ has a relatively simple formulae:

$$
\hat{\beta} = (X^T C^{-1}X)^+X^TC^{-1}\mathbf{Y}
$$

We can replace the $\hat{\beta}$ in the equation above, like so:

$$
r^T\mathbf{C}^{-1}r = \mathbf{Y}^T \mathbf{C}^{-1} \mathbf{Y} - \mathbf{Y}^T \mathbf{C}^{-1} \mathbf{X}(\mathbf{X}^T \mathbf{C}^{-1}\mathbf{X})^+\mathbf{X}^T\mathbf{C}^{-1}\mathbf{Y} - ((\mathbf{X}^T \mathbf{C}^{-1}\mathbf{X})^+\mathbf{X}^T\mathbf{C}^{-1}\mathbf{Y})^T\mathbf{X}^T \mathbf{C}^{-1}\mathbf{Y} + ((\mathbf{X}^T \mathbf{C}^{-1}\mathbf{X})^+\mathbf{X}^T\mathbf{C}^{-1}\mathbf{Y})^T\mathbf{X}^T \mathbf{C}^{-1} \mathbf{X}(\mathbf{X}^T \mathbf{C}^{-1}\mathbf{X})^+\mathbf{X}^T\mathbf{C}^{-1}\mathbf{Y}
$$

This is getting a bit long and complicated. But once again, we can define matrix to simplify the notation a bit. We have $\mathbf{X}\hat{\beta} = \mathbf{X}(\mathbf{X}^T \mathbf{C}^{-1}\mathbf{X})^+\mathbf{X}^T\mathbf{C}^{-1}Y$. We can define a matrix that we will call $\mathbf{K}$, such that $\mathbf{K}=\mathbf{X}(\mathbf{X}^T \mathbf{C}^{-1})^+\mathbf{X}^T\mathbf{C}^{-1}$. That way, we have: $X\hat{\beta} = \mathbf{X}(\mathbf{X}^T \mathbf{C}^{-1}\mathbf{X})^+\mathbf{X}^T\mathbf{C}^{-1}\mathbf{Y} = \mathbf{K}\mathbf{Y}$. In other words, we have re-expressed the part $\mathbf{X}\mathbf{\hat{\beta}}$, as the observed data $\mathbf{Y}$ times some matrix $\mathbf{K}$. And in fact, that matrix $\mathbf{K}$ isn't any matrix, it is a matrix that's used quite often in different ways in linear modelling, because it transforms data $\mathbf{Y}$ into the fitted data: $\mathbf{K}\mathbf{Y}=\mathbf{\hat{Y}}$. So it is yet another quite handy projection matrix (it's not critical here, but just know that you might come across it quite a bit in the context of linear modeling). Based on this reformulation, we can rewrite the long equation from above as:

$$
r^T\mathbf{C}^{-1}r = \mathbf{Y}^T\mathbf{C}^{-1}\mathbf{Y}-\mathbf{Y}^T\mathbf{C}^{-1}\mathbf{K}\mathbf{Y} - \mathbf{Y}^T\mathbf{C}^{-1}\mathbf{K}\mathbf{Y} + \mathbf{Y}^T\mathbf{C}^{-1}\mathbf{K}\mathbf{Y}
$$

In doing so, we see that we have a few terms that cancel out, so we can simplify the equation further:

$$
r^T\mathbf{C}^{-1}r = \mathbf{Y}^T\mathbf{C}^{-1}\mathbf{Y}-\mathbf{Y}^T\mathbf{C}^{-1}\mathbf{K}\mathbf{Y} 
$$

We can simplify it even further, by factorizing some variables that are shared across terms:

$$
r^T\mathbf{C}^{-1}r = \mathbf{Y}^T\mathbf{C}^{-1}(\mathbf{I}-\mathbf{K})\mathbf{Y}
$$

We will once again define a useful matrix to simplify the notation:

$$
\mathbf{P} =\mathbf{C}^{-1}(\mathbf{I}-\mathbf{K}) 
$$

That way, the last part of the model is:

$$
r^T\mathbf{C}^{-1}r = \mathbf{Y}^T\mathbf{P}\mathbf{Y}
$$

And note that the matrix $\mathbf{P}$ can be developped again into:

$$
\begin{align}
\mathbf{P} =& \mathbf{C}^{-1}(I-\mathbf{K}) \\
=& \mathbf{C}^{-1} - \mathbf{C}^{-1}\mathbf{X}(\mathbf{X}^T\mathbf{C}^{-1}\mathbf{X})^+\mathbf{X}^T\mathbf{C}^{-1}
\end{align}
$$

This will be important later.

So now, we can rewrite the restricted likelihood formulae as:

$$
\mathcal{L}(\theta|\mathbf{A}^T\mathbf{Y}) = -\frac{1}{2}log|\mathbf{C}|- \frac{1}{2} log |\mathbf{X}^T \mathbf{C}^{-1} \mathbf{X}| - \frac{1}{2}\mathbf{Y}^T\mathbf{P}\mathbf{Y}
$$

And it is of that function that we have to find the derivative. 


## ReML derivative for vRSA

Note that all we have done above is rewrite the likelihood function into a form that is a bit easier to deal with, such that we can find a derivative that we can work with. What we are after is therefore the derivative of $\mathcal{L}(\theta|\mathbf{A}^T\mathbf{Y})$. Once we find places where the derivative is equal to 0, we know we have reach a local peak of the function. So in other words, what we are after is:

$$
\frac{\delta\mathcal{L}(\theta|\mathbf{A}^T\mathbf{Y})}{\delta \theta} = 0
$$

In the case of a mixed model, or pattern component modelling, the goal is to estimate the covariance matrix $\mathbf{G}$. So we want to find the matrix $\mathbf{G}$ such that the derivative of the likelihood function is equal to 0. And remember that:

$$
\mathbf{C} = \mathbf{Z}\mathbf{G}\mathbf{Z}^T + \sigma^2 \mathbf{I}
$$

In that case, the process would be to derive the likelihood relative to $\mathbf{G}$ and $\sigma^2$ itself. However, in the case of vRSA, we have defined $\mathbf{G}$ itself as:

$$
\mathbf{G} = \sum_{i=1}^k v_i Q_i
$$

We further need to constrain the values of $v_i$, such that it can't be negative, since negative covariance isn't possible. To do so, instead of estimating $v_i$, we actually aim to estimate a latent variable $h_i$, such that:

$$
\mathbf{G} = \sum_{i=1}^k exp(h_i) Q_i
$$

Accordingly, $v_i=log(h_i)$. This ensures that the estimates of $v_i$ are always positive. But we therefore have to differentiate the likelihood function for $h_i$. In other words, what we are after is:

$$
\frac{\delta\mathcal{L}(\theta|\mathbf{A}^T\mathbf{Y})}{\delta h_i} = 0
$$

Now just for the sake of illustration, let's write the full function we are trying to derive when seeking the estimates of $h_i$, by replacing $\mathbf{C}$ by the formulae above:

$$
\mathcal{L}(\theta|\mathbf{A}^T\mathbf{Y}) = -\frac{1}{2}log|\mathbf{Z}\mathbf{\sum_{i=1}^k exp(h_i) Q_i}\mathbf{Z}^T + \sigma^2 \mathbf{I}|- \frac{1}{2} log |X^T (\mathbf{Z}\mathbf{\sum_{i=1}^k exp(h_i) Q_i}\mathbf{Z}^T + \sigma^2 \mathbf{I})^{-1} X| - \frac{1}{2}\mathbf{Y}^T\mathbf{P}\mathbf{Y}
$$

This is quite a difficult formulae to take the derivative from, so we won't get into it here. Here is one solution for it:

$$
\mathcal{L}'(h_i) = -\frac{1}{2}Tr(\mathbf{P}\ exp(h_i) \mathbf{Q_i}) + \frac{1}{2}\mathbf{Y}^T \mathbf{P}\ exp(h_i) \mathbf{Q_i}\mathbf{P} \mathbf{Y}
$$

Unfortunately, this derivative does not have a close formed formulea to find where it is equal to 0, we therefore need to estimate the values of $h_i$ using optimization. In essence, what we need to do is try different values of $h_i$ until we find the combinations that yield 0 for that function. And while we can use the function right above to find the $h_i$ estimates, the function spm_reml_sc uses a slightly different expression of the gradient of the likelihood function, namely:

$$
\mathcal{L}'(h_i) = -\frac{N}{2}Tr(\mathbf{P}\ exp(h_i) \mathbf{Q_i} \mathbf{U}) 
$$

Where

$$
\mathbf{U} = \mathbf{I} - \frac{\mathbf{\mathbf{P}\mathbf{Y}\mathbf{Y}^T}}{N}
$$

Where $N$ is the number of samples. There are a couple of advantages to this formulation, which revolve around computational stability. But more importantly, it means that the observed covariance matrix $\mathbf{Y}\mathbf{Y}^T$ is the only observable we need to estimate our $h_i$ weights, which is why we only pass $\mathbf{Y}\mathbf{Y}^T$ as data to the spm_reml_sc function, as we will describe below.

So long story short, the main objective of the spm_reml_sc function is to find the $h_i$ values such that the function $\mathcal{L}'(h_i) = -\frac{N}{2}Tr(P exp(h_i) Q_i \mathbf{U})$ is equal (or very close) to 0. Note that the derivative of the likelihood returns a vector of value, i.e. the derivative whose dimensions match the number of $Q_i$ component used to model the covariance of the data. The goal is to find the values $h_i$ for which the derivative is equal to 0 along each dimension. And when calculating the derivative for each of the $h_i$, the resulting vector is the direction in which to go to progress towards the local minima of the likelihood function. In case that's unclear, make sure to check out this [youtube tutorial]() series explaining multivariate calculus. 

## Newton-Raphson optimization with Fisher scoring

As stated above, the goal when doing vRSA is to find the values of the covariance components weights $h_i$ that maximize the (restricted) likelihood function, which corresponds to finding the values of $h_i$ such that the derivative of the likelihood function is 0 in all directions. Unfotunately, there is no analytical solution to finding the values of $h_i$ where the derivative is equal to 0, so we have to use some optimization method. One way would be gradient descent: when calculating the derivative of the likelihood for each $h_i$, we get a vector which is the gradient pointing which way is down. And so we can select the next values $h_i$ by increasing each along the direction pointed by the gradient, and repeating until we find the $h_i$ such that the derivative is 0 everywhere, meaning we have arrived at the local maxima of the likelihood function. 

But the question is how big should the increment in the direction of the gradient be? One answer is to use a small fixed increment. This is the standard approach for gradient descent. That would work, but that might not be the most efficient and would require a lot of iterations to converge on the maximum of the likelihood function. But an alternative that is typically more efficient is the so-called Newton-Raphson method, which is also simply referred to as the Newton method. With this approach, the size of the step is taken based on the curvature of the likelihood function. Remember, the derivative gives us a gradient, which tells us which way is up. The curvature is the derivative of the derivative, and what it tells us is, in the direction of the gradient, how fast we are going down. 

The classical analogy to think about it is with a bowl. The important part to get the analogy is to picture the right kind of bowl: if you are thinking of a bowl that is a perfect half sphere, you will get confused by the analogy (at least I did for a very long time). You should think of a breakfast bowl, where the edges are quite steep, but then it flattens out towards the bottom, like in the figure below. Íf we imagine that this breakfast bowl is a likelihood function and that we want to find the bottom of the bowl, if we calculate the derivative at any point of the bowl, it will basically draw and arrow pointing towards the bottom of the bowl. The curvature of the other hand will tell us how steep the slope down is. If we are towards the rim of the bowl, the curvature is typically larger, because at the rim the wall of the bowl is almost straight down. Towards the bottom of the bowl, the curvature is much lower, because the bottom of a bowl is oftentimes almost flat. 

![Breakfast bowl: The curvature is steeper at the rim than at the bottom, because the bottom is essentially flat](../assets/breakfast-bowl.jpg){#fig-correlation_noise_sensitivity fig-alt="Correlation noise sensivity"}

The idea behind the Newton Raphson method is that the size of the step in the direction of the gradient should be taken to reflect the curvature: if the cuvature is low, i.e. the surface is flat, then we can take a larger step because the value of the likelihood doesn't change very fast. In contrast, if the curvature is high, then the values of the likelihood also change fast as a function of the changes in $h_i$ and we want to take smaller steps so as not to overshoot. You can find a very nice description of the approach [here](https://www.youtube.com/watch?v=W7S94pq5Xuo&t=360s). This is the method that is used in the spm_reml_sc function to identify the $h_i$ value that maximize the likelihood function. 

This means we also need to derive the curvature of the likelihood function too. The curvature of the likelihood function is a matrix called the Hessian, which tells us, for each pair of parameters $h_i$ and $h_j$​, how the gradient of the likelihood changes as a function of changes in $h_i$ and $h_j$​. The Hessian is a square matrix of size $k$ by $k$, where $k$ is the number of covariance components (i.e., the number of $h_i$ parameters). The diagonal elements of the Hessian describe the curvature of the likelihood with respect to each individual parameter $h_i$, while the off-diagonal elements describe how the parameters interact with each other.Based on this, we need to determine how big the change in the direction of the gradient should be for each of the $h_i$ values. If we were to stick to the Newton-Raphson method, this would be done by solving the following update equation:

$$
\Delta h = \mathcal{L}''(h_i ,h_j)^{-1} \mathcal{L}'(h_i)
$$

Where:
- $\Delta h$ is the update to the vector of $h_i$ values

This equation tells us that the step size in the Newton-Raphson method is proportional to the inverse of the curvature (Hessian) and the gradient. If the curvature is large (steep slope), the step size will be small, and if the curvature is small (flat slope), the step size will be large. In the case of spm_reml_sc, we are using something called Fisher scoring, which is a variant of the Newton-Raphson method (see a thorough description [here](https://en.wikipedia.org/wiki/Scoring_algorithm)). The key difference is that instead of using the exact Hessian matrix $H$, Fisher scoring uses the expected Hessian, also known as the Fisher information matrix:

$$
\mathcal{I}_{i, j} = -\frac{N}{2}Tr(\mathbf{PQ_i}, \mathbf{PQ_j}) 
$$

The Fisher information matrix is easier to compute than the exact Hessian and is more stable, especially when the likelihood function is noisy or the data are high-dimensional. The update equation for Fisher scoring is:

$$
\Delta h = \mathcal{I}_{i, j}^{-1} \mathcal{L}'(h_i)
$$

So once again, long story short, the way the expected maximization works in spm_reml_sc is based on the Fisher scoring version of the Newton Raphson algorithm, where the next values in the iterative optimization process are derived based on the observed information matrix (as opposed to the exact Hessian). The iterative algorithm calculates the gradient and observed information matrix at given value for the vector or parameters $h$ and selects the values of the next step based on the gradient and the observed curvature. 

## Priors, posterior and variational Bayes

In a traditional ReML scheme, we could have ended here: using the Fisher scoring version of the Newton Raphson method is one way to determine the most likely estimates of $h_i$ given the observed data (restricted to whatever is orthogonal to the fixed effects). However, what we want to do is infer the likelihood of the parameters given the data. The likelihood part only tells us what are the parameters values under which the observed data are the most likely. To go from one to the other, we have the Bayes theorem:

$$
P(\theta|Y) = \frac{P(Y|\theta)P(\theta)}{P{Y}}
$$

Where:
- $P(Y|\theta)$ is the likelihood of the data given the parameters value (the bit that we can maximize following the method layed out above)
- $P(\theta)$ is the prior probability of the parameters $\theta$
- $P(Y)$ is the model likelihood or marginal likelihood
- $P(\theta|Y)$ is the posterior distribution, the probability of the parameter values given the data

In the previous section, I have lied a bit, saying that spm_reml_sc maximizes the ReML using the Raphson-Newton algorithm to obtain the most likely estimates of $h_i$. In fact, it does not maximizes the likelihood function, but instead the free energy, because spm_reml_sc relies on approximate Bayesian inference to determine the probability of the parameters given the data. The goal in Bayesian inference is to estimate the posterior distribution $P(h|\mathbf{Y})$, and one way to do so is relying on variational Laplace. In short, variational Laplace consists in approximating the posterior distribution by assuming that it's going to be somewhat normally distirbuted around it's mode. So the goal is to estimate the mode of the posterior distribution and consider that it is a normal distribution (you can find a more detailed explanation [here](https://alexlepauvre.github.io/variation_laplace_for_dummies/intro.html)). And the way we can find the modes of the posterior distribution is to minimize the free energy, which is an approximation of the model evidence part of the Bayes theorem.

This is a bit of a mouthful, but essentially, instead of trying to minimize the ReML function defined above (i.e. find the values of $h_i$ resulting in a derivative of 0), what we are trying to minimize is the free energy, defined as:

$$
F(h) = log\ P(\mathbf{Y}|h) + log\ P(h) + const
$$

The constant term is meant to control for model complexity. And just as in the case of the (log) ReML function, what we want to do is take the derivative thereof to find the values of $h$ that results in a zero (vector). So we need to define the derivative of the free energy:

$$
\frac{\delta F(h_i)}{\delta h_i} = \frac{\delta ( log\ P(\mathbf{Y}|h) + log\ P(h))}{\delta h_i}
$$

We can separate the formulae as:

$$
\frac{\delta F(h_i)}{\delta h_i} = \frac{\delta (log\ P(\mathbf{Y}|h))}{\delta h_i} +  \frac{\delta (log\ P(h))}{\delta h_i}
$$


We already have the derivative of the log likelihood $log\ P(\mathbf{Y}|h)$, it's the derivative of the ReML function we have described above. Now we need to calculate the derivative of the prior term. Our prior is a multivariate normal distribution of $h$, which is defined:

$$
\begin{align}
P(h) =& \mathcal{N}(h_E, h_C)
=& \frac{1}{(2\pi)^{k/2}|h_C|^{1/2}}exp(-\frac{1}{2}(h - h_E)^Th_C^{-1}(h-h_E))
\end{align}
$$

Since we want the log of that, we have:

$$
log\ P(h) =  -\frac{k}{2} log(2\pi) - \frac{1}{2}log|h_C| - \frac{1}{2}(h-h_E)^T h_C^{-1}(h-h_E)  
$$

The first two terms are a constant, so we can rewrite it as:

$$
log\ P(h) = - \frac{1}{2}(h-h_E)^T h_C^{-1}(h-h_E) + const
$$

And the derivative of the log prior is:

$$
\frac{\delta log\ P(h)}{\delta h_i} = - [h_C^{-1}(h-h_E)]_i
$$

Accordingly, the derivative of the free energy is:

$$
\frac{\delta F(h_i)}{\delta h_i} = -\frac{N}{2}Tr(\mathbf{P}\ exp(h_i) \mathbf{Q_i} \mathbf{U}) - [h_C^{-1}(h-h_E)]_i
$$

This is in fact what spm_reml_sc is optimizing: we are following the free energy gradient to find the estimates of $h_i$ that minimize the free energy. And just in the same way as we can use the Raphson-Newton method with Fisher scoring to find the size of the steps in each iteration, we can also derive the Fisher information matrix for the free energy: 

$$
\frac{\delta F(h_i)}{\delta\delta h_i} = \frac{\delta ( log\ \mathbf{P}(\mathbf{Y}|h) + log\ P(h))}{\delta\delta h_i}
$$

Which simplifies to:

$$
\frac{\delta F(h_i)}{\delta\delta h_i} = -\frac{N}{2}Tr(\mathbf{PQ_i}, \mathbf{PQ_j}) - h_C^{-1}
$$

This is what is happening inside the spm_reml_sc function: we calculate the gradient and the Fisher information matrix of the Free energy function to progress towards the modes and variance of the (normal) posterior distribution of $P(h|\mathbf{Y})$, which is our inference about the values of the weights of the covariance components given the data. In each iteration, we compute how much a change there was in the free energy compared to the previous step. If it's really little, it means we aren't really progressing anymore and that we have reached the best estimates we can, and the optimization process is terminated. 



## Calculating the free energy

Once optimization is done, we still compute the actual free energy of the values we landed on, because in the optimization loop, only the gradient and the predicted free energy were computed, not the free energy itself. This requires calculating the model accuracy as well as the model complexity (which is the constant term we sidestepped during the optimization process). In other words:

$$
F = Accuracy - Complexity
$$

The accuracy is the Expected log likelihood of the data under the posterior. The Complexity is hte KL divergence between the posterior and the prior. We won't go into the full detail of the derivation of the free energy formulae (see [here](https://alexlepauvre.github.io/variation_laplace_for_dummies/intro.html) for an extended treatment on that subject), but here is what each component looks like in our case:

$$
Complexity = \frac{1}{2}(log|Ph|-log|h_C^{-1}|-e^ThPe+tr(Ph^-1)h_C^{-1}-k)
$$

Where:
- $Ph=C_h^{-1}$ is the posterior precision (i.e. inverse of the posterior covariane)
- $e=h-h_E$ is the difference between the estimated and prior values for the mean values of the covariance component weights
- $k$ is the number of hyperparameters.

$$
Accuracy = -\frac{N}{2} log |C| -\frac{N}{2}log|X^TC^{-1}X| - \frac{1}{2}tr(PYY^T)| 
$$


## Conclusion 
And with that, we have explained all the key components of spm_reml_sc. To summarize the entire approach of spm_reml_sc in a couple of sentences: spm_reml_sc approximates the posterior distribution of the $h$ weights of the covariance components that approximate the covariance structure of the data the best by ascending the free energy landscape, using the Newton Raphson optimization method with Fisher scoring. The components of the free energy function that matter for the optimization consist of the log of the restricted maximum likelihood function and of the prior distribution, which is a multivariate normal distribution. The script relies on a derivation of the derivative and of the curvature of both these terms (likelihood and priors) that are then calculated iteratively based on novel estimates of the $h$ values, which determines the values of $h$ taken in the next step until the predicted difference in free energy from one iteration to the next becomes low, which indicates that the estimates maximizing the free energy function have been found. 

In the final notebook of this series, we will describe step by step where each of these steps are taken in the spm_reml_sc function. 